# Step 9 — EgoVLP Training (MLP & Transformer)

Trains MLP and Transformer classifiers on the 256-dim EgoVLP features extracted in Step 6.
All model logic lives in the `.py` files; this notebook only runs CLI commands.

**Prerequisites:** EgoVLP features must be extracted in `FEATURES_DIR_EGOVLP`  
**Output:** checkpoints in `MY_MODELS_DIR/error_recognition/{MLP,Transformer}/egovlp/`

In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Path constants (FIXED — do not modify) ─────────────────────────────────
DRIVE_ROOT          = '/content/drive/MyDrive/AML_Project'
FEATURES_DIR_EGOVLP = f'{DRIVE_ROOT}/CaptainCook4D/features_egovlp'
MY_MODELS_DIR       = f'{DRIVE_ROOT}/models'
REPO_DIR            = '/content/code'
print('Paths defined.')

In [ ]:
# ── 3. Clone / update repo ────────────────────────────────────────────────────
import os, sys

if not os.path.exists(REPO_DIR):
    !git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
!pip install -q wandb torcheval scikit-learn tqdm
import torch
print(f'torch {torch.__version__} | CUDA {torch.cuda.is_available()}')

In [ ]:
# ── 5. Disable wandb + sanity check ──────────────────────────────────────────
# Disabling wandb avoids login prompts; remove this line if you want to log runs.
os.environ['WANDB_MODE'] = 'disabled'

import pathlib, numpy as np
from constants import Constants as const
from core.config import Config
from core.models.blocks import fetch_input_dim
from base import fetch_model

# Verify feature files are available
egovlp_dir = pathlib.Path(FEATURES_DIR_EGOVLP)
npz_files  = list(egovlp_dir.glob('*.npz'))
print(f'EgoVLP .npz files found: {len(npz_files)}')
if npz_files:
    sample = np.load(str(npz_files[0]))
    print(f'Sample shape: {sample["arr_0"].shape}')   # expected: (N_segs, 256)

# Verify model factory works for EgoVLP
conf = Config()
conf.backbone = const.EGOVLP
conf.variant  = const.MLP_VARIANT
conf.modality = ['video']
conf.device   = 'cpu'
print(f'Input dim for EgoVLP: {fetch_input_dim(conf)}')   # expected: 256
print(f'Model: {fetch_model(conf)}')                       # expected: MLP(256→512→1)

## Training — MLP

Hyperparameters (paper Appendix B.3, non-sequential models):
- `lr=1e-3`, `weight_decay=1e-3`, `num_epochs=50`, `pos_weight=1.5` (fixed in `base.py`)

In [ ]:
# ── 6. Train MLP — step split ─────────────────────────────────────────────────
!python train_er.py \
    --variant   MLP \
    --backbone  egovlp \
    --split     step \
    --num_epochs 50 \
    --lr        1e-3 \
    --weight_decay 1e-3 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt_directory {MY_MODELS_DIR}

In [ ]:
# ── 7. Train MLP — recordings split ──────────────────────────────────────────
!python train_er.py \
    --variant   MLP \
    --backbone  egovlp \
    --split     recordings \
    --num_epochs 50 \
    --lr        1e-3 \
    --weight_decay 1e-3 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt_directory {MY_MODELS_DIR}

## Training — Transformer (ErFormer)

Same hyperparameters as MLP.  
Note: with 256-dim EgoVLP input, the modality-split branches in `ErFormer.forward()`
(`if dim // 1024 == 1`) are never triggered — the full 256-dim encoding goes
straight to the MLP decoder. This is correct behaviour for a single-modality backbone.

In [ ]:
# ── 8. Train Transformer — step split ────────────────────────────────────────
!python train_er.py \
    --variant   Transformer \
    --backbone  egovlp \
    --split     step \
    --num_epochs 50 \
    --lr        1e-3 \
    --weight_decay 1e-3 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt_directory {MY_MODELS_DIR}

In [ ]:
# ── 9. Train Transformer — recordings split ───────────────────────────────────
!python train_er.py \
    --variant   Transformer \
    --backbone  egovlp \
    --split     recordings \
    --num_epochs 50 \
    --lr        1e-3 \
    --weight_decay 1e-3 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt_directory {MY_MODELS_DIR}

## Quick eval — verify checkpoints

Run after each training cell to check the best checkpoint was saved.
Full comparative analysis (vs Omnivore) is in Step 10.

In [ ]:
# ── 10. List saved checkpoints ────────────────────────────────────────────────
import pathlib
ckpt_root = pathlib.Path(MY_MODELS_DIR) / 'error_recognition'
for p in sorted(ckpt_root.rglob('*egovlp*_best.pt')):
    print(p)

In [ ]:
# ── 11. Quick global eval — MLP step split ────────────────────────────────────
MLP_STEP_CKPT = f'{MY_MODELS_DIR}/error_recognition/MLP/egovlp/error_recognition_step_egovlp_MLP_video_best.pt'

!python -m core.evaluate \
    --variant   MLP \
    --backbone  egovlp \
    --split     step \
    --threshold 0.6 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt      {MLP_STEP_CKPT}

In [ ]:
# ── 12. Quick global eval — Transformer step split ───────────────────────────
TR_STEP_CKPT = f'{MY_MODELS_DIR}/error_recognition/Transformer/egovlp/error_recognition_step_egovlp_Transformer_video_best.pt'

!python -m core.evaluate \
    --variant   Transformer \
    --backbone  egovlp \
    --split     step \
    --threshold 0.6 \
    --segment_features_directory {FEATURES_DIR_EGOVLP} \
    --ckpt      {TR_STEP_CKPT}